# Quickstart: Spherical Extrapolation By Date

This notebook downloads one full-disk HMI vector observation, builds a simple spherical config, and runs the extrapolation.

Workflow:
1. Edit the parameter cell below.
2. Run all cells.
3. Open the exported result from the output directory.

In [ ]:
# Edit only this cell
JSOC_EMAIL = "your_email@example.com"
DATE = "2024-05-07T17:24:00"
OUTPUT_NAME = "spherical_quickstart_20240507"
EPOCHS = 20
EXPORT_FORMATS = ["vtk", "npz"]

In [ ]:
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import drms

from nf2.core.runner import run
from nf2.data.download import donwload_ds
from nf2.export.core import export_checkpoint
from nf2.train.util import load_yaml_config

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

timestamp = datetime.fromisoformat(DATE)
download_root = repo_root / "data" / "notebooks" / OUTPUT_NAME
full_disk_dir = download_root / "full_disk"
full_disk_dir.mkdir(parents=True, exist_ok=True)

client = drms.Client(email=JSOC_EMAIL)
dataset = f"hmi.B_720s[{timestamp.isoformat('_', timespec='seconds')}]"
process = {"HmiB2ptr": {"l": 1}}
donwload_ds(dataset, str(full_disk_dir), client, process=process)

br_files = sorted(full_disk_dir.glob("*Br.fits"))
bt_files = sorted(full_disk_dir.glob("*Bt.fits"))
bp_files = sorted(full_disk_dir.glob("*Bp.fits"))
assert br_files and bt_files and bp_files, "Converted Br/Bt/Bp FITS files were not found in the download directory."

config = load_yaml_config(str(repo_root / "config" / "spherical" / "377_spherical.yaml"))
config["base_path"] = str(repo_root / "results" / "notebooks" / OUTPUT_NAME)
config["work_directory"] = str(repo_root / "work" / "notebooks" / OUTPUT_NAME)
config["logging"] = deepcopy(config.get("logging", {}))
config["logging"]["offline"] = True
config["training"] = deepcopy(config.get("training", {}))
config["training"]["epochs"] = EPOCHS
config["data"] = deepcopy(config.get("data", {}))
config["data"]["train_configs"] = [{
    "type": "map",
    "ds_id": "map",
    "files": {"Br": str(br_files[0]), "Bt": str(bt_files[0]), "Bp": str(bp_files[0])},
}]

config

In [ ]:
run(**config)

In [ ]:
checkpoint_path = Path(config["base_path"]) / "extrapolation_result.nf2"
exports = {fmt: export_checkpoint(str(checkpoint_path), fmt) for fmt in EXPORT_FORMATS}
checkpoint_path, exports